# Filterdiagnostik für Transaktionsdaten

Dieses Notebook vergleicht die ungefilterten Transaktionen in `data/interim/transactions_per_year` mit dem gefilterten Output in `data/interim/transactions_per_year_filtered` (<- hier noch separate Transaktionen).

Es zeigt die Anzahl der Reihen, Produkte und Filialen, den Effekt der einzelnen Filter und die aktivsten Produkte, Filialen, Mandanten und Warengruppen im gefilterten Datensatz. Externe Produkte, die weder als FCM noch als Pseudo klassifiziert sind, werden ausgeschlossen.


## Setup

Die Filterbedingungen werden direkt aus `src.data.cleaning.rules` importiert, damit die Auswertung dieselbe Logik wie die Pipeline verwendet.


In [10]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.common import read_parquet_expr
from src.data.cleaning.rules import (
    ALLOWED_FCM_ARTICLE_IDS,
    ALLOWED_MANDANT_IDS,
    EXTERNAL_PRODUCT_RULE,
    FCM_RULE,
    MANDANT_RULE,
    MIN_UMS_MENGE,
    PSEUDO_ARTICLE_IDS,
    WEIGHT_CONTENT_LIKE,
    WEIGHT_RULE,
    fcm_filter_condition,
    fcm_or_pseudo_filter_condition,
    mandant_filter_condition,
    transaction_filter_condition,
    ums_menge_filter_condition,
    weight_filter_condition,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

IN_DIR = ROOT / "data" / "interim" / "transactions_per_year"
OUT_DIR = ROOT / "data" / "interim" / "transactions_per_year_filtered"
IN_GLOB = IN_DIR / "transactions_year_*.parquet"
OUT_GLOB = OUT_DIR / "transactions_year_*.parquet"

if not list(IN_DIR.glob("transactions_year_*.parquet")):
    raise FileNotFoundError(f"Keine Parquet-Dateien in {IN_DIR} gefunden")
if not list(OUT_DIR.glob("transactions_year_*.parquet")):
    raise FileNotFoundError(f"Keine Parquet-Dateien in {OUT_DIR} gefunden")

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("SET preserve_insertion_order=false")

raw_expr = read_parquet_expr(IN_GLOB)
filtered_expr = read_parquet_expr(OUT_GLOB)

print(f"IN_DIR:  {IN_DIR}")
print(f"OUT_DIR: {OUT_DIR}")


IN_DIR:  /Users/vlada/UNI/SoSe2026/ba/ba_code/data/interim/transactions_per_year
OUT_DIR: /Users/vlada/UNI/SoSe2026/ba/ba_code/data/interim/transactions_per_year_filtered


## Filterstatus und technische Bedingungen

Diese Tabelle zeigt, welche Filter aktiv sind und welche konkrete SQL-Bedingung aus der Pipeline verwendet wird. Die fachliche Bedeutung der Filter steht separat unter der Tabelle.


In [11]:
filter_definitions = pd.DataFrame([
    {
        "Filter": "UMS_MENGE",
        "aktiv": True,
        "Bedingung": ums_menge_filter_condition(),
        "Parameter": f"MIN_UMS_MENGE = {MIN_UMS_MENGE}",
    },
    {
        "Filter": "MANDANT_ID",
        "aktiv": MANDANT_RULE,
        "Bedingung": mandant_filter_condition() if MANDANT_RULE else "deaktiviert",
        "Parameter": f"erlaubte Mandanten = {sorted(ALLOWED_MANDANT_IDS)}" if MANDANT_RULE else "-",
    },
    {
        "Filter": "FCM-Artikel",
        "aktiv": FCM_RULE,
        "Bedingung": fcm_filter_condition() if FCM_RULE else "deaktiviert",
        "Parameter": f"{len(ALLOWED_FCM_ARTICLE_IDS):,} erlaubte Artikel-IDs" if FCM_RULE else "-",
    },
    {
        "Filter": "Externe Artikel",
        "aktiv": EXTERNAL_PRODUCT_RULE,
        "Bedingung": fcm_or_pseudo_filter_condition() if EXTERNAL_PRODUCT_RULE else "deaktiviert",
        "Parameter": (
            f"{len(ALLOWED_FCM_ARTICLE_IDS | PSEUDO_ARTICLE_IDS):,} FCM-/Pseudo-Artikel-IDs"
            if EXTERNAL_PRODUCT_RULE else "-"
        ),
    },
    {
        "Filter": "ARTIKEL_INHALT Gewicht",
        "aktiv": WEIGHT_RULE,
        "Bedingung": weight_filter_condition() if WEIGHT_RULE else "deaktiviert",
        "Parameter": f"ARTIKEL_INHALT LIKE {WEIGHT_CONTENT_LIKE!r}" if WEIGHT_RULE else "-",
    },
])

filter_definitions


,Filter,aktiv,Bedingung,Parameter
0,UMS_MENGE,True,"""UMS_MENGE"" > 0.01",MIN_UMS_MENGE = 0.01
1,MANDANT_ID,True,"""MANDANT_ID"" IN (110, 130, 135)","erlaubte Mandanten = [110, 130, 135]"
2,FCM-Artikel,True,"""ARTIKEL_ID"" IN (560039, 579781, 1376569, 1376...",103 erlaubte Artikel-IDs
3,ARTIKEL_INHALT Gewicht,True,"(CASE WHEN LOWER(COALESCE(CAST(""ARTIKEL_INHALT...",ARTIKEL_INHALT LIKE '%amm%'


## Fachliche Bedeutung der Filter

- `UMS_MENGE`: Behält nur Transaktionszeilen mit einer relevanten positiven Verkaufsmenge. Dadurch werden Nullmengen, Kleinstmengen unterhalb des Schwellwerts und negative Mengen entfernt.
- `MANDANT_ID`: Beschränkt die Daten auf die für die Analyse vorgesehenen Mandanten.
- `FCM-Artikel`: Beschränkt die Daten auf die definierte FCM-Artikelliste und damit auf die untersuchte Produktauswahl.
- `Externe Artikel`: Behält ausschließlich Produkte aus der FCM- oder Pseudo-Artikelliste. Produkte, die zu keiner der beiden Listen gehören, werden vollständig aus dem Datensatz entfernt.
- `ARTIKEL_INHALT Gewicht`: Behält Produkte, bei denen eine Abverkaufsmenge in kg ableitbar ist. Die Filterregel prüft dafür die Inhaltsangabe `ARTIKEL_INHALT` auf Gramm-/Kilogramm-Angaben. Im gefilterten Output wird `ABVERKAUFTE_MENGE_KG` anschließend aus `GRAMM_BON` gebildet: Für echte Gewichtsartikel (`GEWICHTSARTIKEL = 1`) wird `GRAMM_BON` direkt verwendet; bei verpackten Artikeln wird `GRAMM_BON` nur verwendet, wenn `ARTIKEL_INHALT` eine Gewichtseinheit wie `kg`, `Kilogramm`, `g`, `gr.` oder `Gramm` enthält. Für andere Artikel wäre keine belastbare kg-Menge ableitbar.


## Umfang vor und nach dem Filtern

Eine Reihe ist hier eine eindeutige Kombination aus `ARTIKEL_ID` und `MARKT_ID`.


In [12]:
def where_clause(condition):
    return f"WHERE {condition}" if condition else ""


def count_snapshot(expr, condition=None):
    where_sql = where_clause(condition)
    return con.execute(
        f"""
        SELECT
            COUNT(*)::BIGINT AS Zeilen,
            COUNT(*) FILTER (WHERE ARTIKEL_ID IS NOT NULL AND MARKT_ID IS NOT NULL)::BIGINT AS Zeilen_mit_Reihenschluessel,
            COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID))::BIGINT AS Reihen,
            COUNT(DISTINCT ARTIKEL_ID)::BIGINT AS Produkte,
            COUNT(DISTINCT MARKT_ID)::BIGINT AS Filialen,
            MIN(CAST(DATE AS DATE)) AS erster_Tag,
            MAX(CAST(DATE AS DATE)) AS letzter_Tag
        FROM {expr}
        {where_sql}
        """
    ).fetchdf().iloc[0].to_dict()


overview = pd.DataFrame(
    [
        {"Datensatz": "ungefiltert", **count_snapshot(raw_expr)},
        {"Datensatz": "gefiltert", **count_snapshot(filtered_expr)},
    ]
)
overview["entfernte_Zeilen"] = overview["Zeilen"].iloc[0] - overview["Zeilen"]
overview["behaltener_Zeilenanteil_%"] = (100 * overview["Zeilen"] / overview["Zeilen"].iloc[0]).round(2)

overview


,Datensatz,Zeilen,Zeilen_mit_Reihenschluessel,Reihen,Produkte,Filialen,erster_Tag,letzter_Tag,entfernte_Zeilen,behaltener_Zeilenanteil_%
0,ungefiltert,92041082,92041082,159839,3783,113,2021-07-11,2026-07-10,0,100.00
1,gefiltert,497752,497752,3667,49,106,2025-04-17,2026-07-10,91543330,0.54


## Effekt der einzelnen Filter

Die Filter werden sequenziell ausgewertet. `entfernt_*` bedeutet daher: zusätzlich entfernt nach allen vorherigen Filtern.


In [13]:
def combine_conditions(left, right):
    return f"({left}) AND ({right})" if left else right


filter_steps = [("UMS_MENGE", ums_menge_filter_condition())]
if MANDANT_RULE:
    filter_steps.append(("MANDANT_ID", mandant_filter_condition()))
if FCM_RULE:
    filter_steps.append(("FCM-Artikel", fcm_filter_condition()))
if EXTERNAL_PRODUCT_RULE:
    filter_steps.append(("Externe Artikel", fcm_or_pseudo_filter_condition()))
if WEIGHT_RULE:
    filter_steps.append(("ARTIKEL_INHALT Gewicht", weight_filter_condition()))

impact_rows = []
current_condition = None
previous = count_snapshot(raw_expr)

for filter_name, filter_condition in filter_steps:
    current_condition = combine_conditions(current_condition, filter_condition)
    current = count_snapshot(raw_expr, current_condition)
    row = {
        "Filter": filter_name,
        "vorher_Zeilen": previous["Zeilen"],
        "nachher_Zeilen": current["Zeilen"],
        "entfernt_Zeilen": previous["Zeilen"] - current["Zeilen"],
        "entfernt_Zeilen_%": round(
            100 * (previous["Zeilen"] - current["Zeilen"]) / previous["Zeilen"], 2
        ) if previous["Zeilen"] else 0,
        "vorher_Reihen": previous["Reihen"],
        "nachher_Reihen": current["Reihen"],
        "entfernt_Reihen": previous["Reihen"] - current["Reihen"],
        "vorher_Produkte": previous["Produkte"],
        "nachher_Produkte": current["Produkte"],
        "entfernt_Produkte": previous["Produkte"] - current["Produkte"],
        "vorher_Filialen": previous["Filialen"],
        "nachher_Filialen": current["Filialen"],
        "entfernt_Filialen": previous["Filialen"] - current["Filialen"],
    }
    impact_rows.append(row)
    previous = current

filter_impact = pd.DataFrame(impact_rows)
expected_kept = count_snapshot(raw_expr, transaction_filter_condition())
actual_kept = count_snapshot(filtered_expr)

if expected_kept["Zeilen"] != actual_kept["Zeilen"]:
    print(
        "WARNUNG: Die erwartete Zeilenzahl nach Filterlogik weicht vom vorhandenen OUT_DIR ab: "
        f"erwartet={expected_kept['Zeilen']:,}, OUT_DIR={actual_kept['Zeilen']:,}"
    )

filter_impact


,Filter,vorher_Zeilen,nachher_Zeilen,entfernt_Zeilen,entfernt_Zeilen_%,vorher_Reihen,nachher_Reihen,entfernt_Reihen,vorher_Produkte,nachher_Produkte,entfernt_Produkte,vorher_Filialen,nachher_Filialen,entfernt_Filialen
0,UMS_MENGE,92041082,92007719,33363,0.04,159839,159796,43,3783,3778,5,113,113,0
1,MANDANT_ID,92007719,92007719,0,0.00,159796,159796,0,3778,3778,0,113,113,0
2,FCM-Artikel,92007719,497752,91509967,99.46,159796,3667,156129,3778,49,3729,113,106,7
3,ARTIKEL_INHALT Gewicht,497752,497752,0,0.00,3667,3667,0,49,49,0,106,106,0


## Aktivste Entitäten im gefilterten Datensatz

Aktivität wird für Produkte, Filialen, Mandanten und Warengruppen über die Anzahl der Transaktionszeilen gemessen. Ergänzend werden Tage, Produkte, Filialen, Menge und Umsatz ausgewiesen, soweit sie für die jeweilige Entität sinnvoll sind.


In [14]:
def top_entities(group_cols, label_cols=None, top_n=10, order_by="Zeilen DESC"):
    label_cols = label_cols or []
    select_cols = []
    group_sql = []
    for col in group_cols:
        select_cols.append(col)
        group_sql.append(col)
    for col in label_cols:
        select_cols.append(f"arg_max({col}, DATE) AS {col}")

    select_sql = ",\n            ".join(select_cols)
    group_by_sql = ", ".join(group_sql)

    return con.execute(
        f"""
        SELECT
            {select_sql},
            COUNT(*)::BIGINT AS Zeilen,
            COUNT(DISTINCT CAST(DATE AS DATE))::BIGINT AS Nachfragetage,
            COUNT(DISTINCT ARTIKEL_ID)::BIGINT AS Produkte,
            COUNT(DISTINCT MARKT_ID)::BIGINT AS Filialen,
            SUM(COALESCE(UMS_MENGE, 0.0))::DOUBLE AS Summe_UMS_MENGE,
            SUM(COALESCE(UMS_VK_WERT, 0.0))::DOUBLE AS Summe_UMS_VK_WERT
        FROM {filtered_expr}
        GROUP BY {group_by_sql}
        ORDER BY {order_by}
        LIMIT {top_n}
        """
    ).fetchdf()


def display_top(title, dataframe):
    print(title)
    display(dataframe)


In [15]:
top_products = top_entities(
    group_cols=["ARTIKEL_ID"],
    label_cols=["ARTIKEL_BEZ", "ARTIKEL_INHALT", "VERKAUFSEINHEIT", "GEWICHTSARTIKEL", "WGR_ID", "N_WARENKLASSE_KBEZ"],
    order_by="Zeilen DESC",
)
display_top("Aktivste Produkte nach Transaktionszeilen", top_products)


Aktivste Produkte nach Transaktionszeilen


,ARTIKEL_ID,ARTIKEL_BEZ,ARTIKEL_INHALT,VERKAUFSEINHEIT,GEWICHTSARTIKEL,WGR_ID,N_WARENKLASSE_KBEZ,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,1382765,Braunschweiger feine Streichmettwurst,125 Gramm,St,0,900,"Streichfähige Rohwurst (Mett-, Streichmett- un...",59572,343,1,106,71199.000,113826.6799
1,1382777,Goldmarie gekochte Schweinemettwurst,1 Kilogramm,kg,1,900,"Streichfähige Rohwurst (Mett-, Streichmett- un...",53367,309,1,105,5321.392,75265.4400
2,1382764,Braunschweiger grobe Streichmettwurst,125 Gramm,St,0,900,"Streichfähige Rohwurst (Mett-, Streichmett- un...",53024,345,1,106,62232.000,96309.2607
3,1382768,Fleischkäse ofengebacken,1 Kilogramm,kg,1,900,Fleischkäseprodukte,48864,382,1,106,14473.782,166031.5007
4,1382767,Fleischwurst ohne Knoblauch,1 Kilogramm,kg,1,900,Fleischwurst,48277,374,1,105,11836.015,161927.0800
5,1382766,Fleischwurst mit Knoblauch,1 Kilogramm,kg,1,900,Fleischwurst,44193,367,1,105,10925.711,149890.5801
6,1376860,Pfefferbeißer,1 Kilogramm,kg,1,900,"Rohwurst, Stückware",29592,298,1,105,6067.708,111967.6400
7,1406063,SB Fleischkäse roh in Schale,1 Kilogramm,kg,1,900,Fleischkäseprodukte,20178,210,1,105,9806.779,67706.8500
8,1382788,"Goldmarie Friesen Bratwurst, gebrüht",1 Kilogramm,kg,1,900,Bratwurst,15592,249,1,105,14974.304,73599.1012
9,1413120,Schweinebraten-Aufschnitt Gyros Art,1 Kilogramm,kg,1,900,"Bratenaufschnitt, Rind-, Schwein",13627,145,1,100,1605.464,29003.6800


In [16]:
top_stores = top_entities(
    group_cols=["MARKT_ID"],
    label_cols=["MARKT_NR", "MANDANT_ID", "LEH_SEH"],
)
display_top("Aktivste Filialen", top_stores)


Aktivste Filialen


,MARKT_ID,MARKT_NR,MANDANT_ID,LEH_SEH,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,1100016,16,110,LEH,19274,353,41,1,7825.4596,61361.7611
1,1300024,24,130,LEH,11035,358,39,1,5325.9271,32640.9803
2,1350051,51,135,SEH,10922,340,24,1,5156.5367,28593.1297
3,1300019,19,130,LEH,10407,352,30,1,3887.6903,29015.4901
4,1300023,23,130,LEH,10264,358,34,1,5226.3342,27029.4401
5,1300003,3,130,LEH,9781,361,37,1,4117.5699,30788.7199
6,1300021,21,130,LEH,9741,348,36,1,4908.7573,25389.2899
7,1100074,74,110,LEH,8300,353,41,1,3075.0137,21274.0000
8,1300002,2,130,LEH,8137,355,40,1,3319.4598,22940.9200
9,1350072,72,135,SEH,8035,303,23,1,3279.1949,21278.0799


In [17]:
top_mandants = top_entities(
    group_cols=["MANDANT_ID"],
    label_cols=["LEH_SEH"],
)
display_top("Aktivste Mandanten", top_mandants)


Aktivste Mandanten


,MANDANT_ID,LEH_SEH,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,110,LEH,273568,421,46,64,142981.0410,849540.2919
1,130,LEH,139728,371,45,20,65753.0068,404904.0396
2,135,SEH,84456,371,41,22,38145.1554,230372.5893


In [18]:
top_warengruppen = top_entities(
    group_cols=["WGR_ID"],
    label_cols=["N_WARENKLASSE_KBEZ"],
)
display_top("Aktivste Warengruppen", top_warengruppen)


Aktivste Warengruppen


,WGR_ID,N_WARENKLASSE_KBEZ,Zeilen,Nachfragetage,Produkte,Filialen,Summe_UMS_MENGE,Summe_UMS_VK_WERT
0,900,Fleischwurst,472729,419,26,106,224007.4378,1.265437e+06
1,890,Schweinefleisch,25023,289,23,106,22871.7654,2.193800e+05


## Hinweise

Die Filterwirkung ist sequenziell berechnet. Wenn ein Datensatz bereits durch einen früheren Filter entfernt wurde, wird er bei späteren Filtern nicht erneut gezählt.
